In [0]:
from datetime import date, timedelta
from pyspark.sql.functions import *

yesterday = date.today() - timedelta(days=1)
yesterday_str = yesterday.strftime('%Y%m%d')
#print(yesterday_str)

dbutils.widgets.text("file_run_date", yesterday_str)
#dbutils.widgets.text("file_run_date", "20260207") #as running with data as of 20260207
file_run_date = dbutils.widgets.get("file_run_date")

#dbutils.widgets.remove("file_run_date")
print(file_run_date)

## **delete the data in case of re run has to be done**

In [0]:
%sql
delete from `spark-databricks-project-catalog`.bronze.online_orders where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.bronze.online_customers where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.bronze.online_products where file_date = :file_run_date;

delete from `spark-databricks-project-catalog`.bronze.offline_orders where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.bronze.offline_customers where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.bronze.offline_products where file_date = :file_run_date;

## The above delete can be done in Python too

In [0]:
%skip
tables = ['online_orders', 'online_customers', 'online_products', 'offline_orders', 'offline_customers', 'offline_products']

for table in tables:
    spark.sql(f"delete from `spark-databricks-project-catalog`.bronze.{table} where file_date = '{file_run_date}'")

## Load Online Data

In [0]:
#orders table
df = spark.read.format("csv") \
     .option("header", True) \
     .load(f"s3://spark-databricks-project/orders/online data/{file_run_date}/online_orders.csv")

df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("online")) \
             .withColumn("ingest_ts",current_timestamp())
#df_audit.printSchema()

df_audit.write.mode("append") \
        .saveAsTable("`spark-databricks-project-catalog`.bronze.online_orders")

#df = spark.table("`spark-databricks-project-catalog`.bronze.online_orders") 
#df.show()       

In [0]:
#customers table
df = spark.read.format("csv") \
               .option("header", True) \
               .load(f"s3://spark-databricks-project/orders/online data/{file_run_date}/online_customers*")

df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("online")) \
             .withColumn("ingest_ts",current_timestamp())

df_audit.write.mode("append") \
        .saveAsTable("`spark-databricks-project-catalog`.bronze.online_customers") 

#df = spark.table("`spark-databricks-project-catalog`.bronze.online_customers") 
#df.show(10)                              

In [0]:
#products table
df = spark.read.format("csv") \
               .option("header",True) \
               .load(f"s3://spark-databricks-project/orders/online data/{file_run_date}/online_products*")

df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("online")) \
             .withColumn("ingest_ts",current_timestamp())

df_audit.write.mode("append") \
              .saveAsTable("`spark-databricks-project-catalog`.bronze.online_products")    
                   
#df = spark.table("`spark-databricks-project-catalog`.bronze.online_products")
#df.show(10)


## Load Offline Data

In [0]:
#offline orders table
df = spark.read.format("csv") \
               .option("header",True) \
               .load(f"s3://spark-databricks-project/orders/offline data/{file_run_date}/offline_orders.csv")
              
df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("offline")) \
             .withColumn("ingest_ts",current_timestamp())

df_audit.write.mode("append") \
              .saveAsTable("`spark-databricks-project-catalog`.bronze.offline_orders")

# df = spark.table("`spark-databricks-project-catalog`.bronze.offline_orders")
# df.show(10)
                       

In [0]:
#customer table
df = spark.read.format("csv") \
               .option("header",True) \
               .load(f"s3://spark-databricks-project/orders/offline data/{file_run_date}/offline_customers.csv")
              
df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("offline")) \
             .withColumn("ingest_ts",current_timestamp())

df_audit.write.mode("append") \
              .saveAsTable("`spark-databricks-project-catalog`.bronze.offline_customers")

# df = spark.table("`spark-databricks-project-catalog`.bronze.offline_customers")  
# df.show(10)           

In [0]:
#products table
df = spark.read.format("csv") \
               .option("header",True) \
               .load(f"s3://spark-databricks-project/orders/offline data/{file_run_date}/offline_products.csv")
              
df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source", lit("offline")) \
             .withColumn("ingest_ts", current_timestamp())
             
df_audit.write.mode("append") \
        .saveAsTable("`spark-databricks-project-catalog`.bronze.offline_products")               